# 10 · Dense sparse and hybrid retrieval

## The engineering question
Which misses are repaired by lexical matching, learned expansion, or rank fusion?

**Learning outcomes:** compare dense, BM25, SPLADE, and fusion on identical eligible sources; inspect term expansion; hand-check reciprocal rank fusion; distinguish score scales; retain misses instead of selecting only wins.

Dense encoders compress text into a fixed-width vector. BM25 matches token statistics. SPLADE produces sparse vocabulary weights using a learned masked-language model and pooling; expansion can retrieve words absent from the input but can also introduce noise. We explicitly prune to 128 nonzero weights. Fusion combines ranked lists without pretending their raw scores have the same meaning.

The transparent fusion below uses one-based ranks and constant 60: each appearance contributes 1/(60 + rank). Qdrant's pinned built-in RRF has its own defaults; it is reported as a separate system. The experiment uses all current public course documents and development queries, k=5 and twenty candidates per branch. No test-set parameter selection occurs here.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("10", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
import time
from uuid import uuid4
from vector_masterclass.data import documents, queries, validate_gold
from vector_masterclass.chunking import chunk_document
from vector_masterclass.store import VectorIndex
from vector_masterclass.retrieval import LexicalIndex, reciprocal_rank_fusion
from vector_masterclass.metrics import ranking_metrics
from vector_masterclass.models import sparse_embed, sparse_model

assert validate_gold()["human_review"] == "approved"
docs = documents(tenant="public")
chunks = [c for d in docs for c in chunk_document(d)]
lookup = {c.id: c for c in chunks}
lexical = LexicalIndex([c.text for c in chunks], list(lookup))
index = VectorIndex("vmc_10_" + uuid4().hex[:8])
index.create()
rows, rankings = [], []
started = time.perf_counter()
try:
    for doc in docs:
        index.upsert(chunk_document(doc), document_id=doc.id)
    ingest_seconds = time.perf_counter() - started
    for q in queries("development"):
        lists = {}
        for mode in ["dense", "sparse", "hybrid"]:
            started = time.perf_counter()
            lists[mode] = [
                h.id for h in index.search(q.text, tenant=q.tenant, mode=mode, k=20, exact=True)
            ]
            rows.append(
                {
                    "query": q.id,
                    "answerable": q.answerable,
                    "method": mode,
                    "seconds": time.perf_counter() - started,
                    **ranking_metrics(
                        [lookup[i].document_key for i in lists[mode]], q.relevance, 5
                    ),
                }
            )
        started = time.perf_counter()
        lists["BM25"] = [i for i, s in lexical.search(q.text, k=20)]
        rows.append(
            {
                "query": q.id,
                "answerable": q.answerable,
                "method": "BM25",
                "seconds": time.perf_counter() - started,
                **ranking_metrics([lookup[i].document_key for i in lists["BM25"]], q.relevance, 5),
            }
        )
        lists["RRF60 dense+BM25"] = [
            i for i, s in reciprocal_rank_fusion([lists["dense"], lists["BM25"]])
        ]
        rows.append(
            {
                "query": q.id,
                "answerable": q.answerable,
                "method": "RRF60 dense+BM25",
                "seconds": None,
                **ranking_metrics(
                    [lookup[i].document_key for i in lists["RRF60 dense+BM25"]], q.relevance, 5
                ),
            }
        )
        for method, ids in lists.items():
            rankings.append(
                {
                    "query": q.id,
                    "question": q.text,
                    "method": method,
                    "top_documents": list(dict.fromkeys(lookup[i].document_key for i in ids))[:5],
                }
            )
finally:
    index.drop()
    index.close()
results = lesson.table(pd.DataFrame(rows))
display(results.groupby("method")[["recall", "mrr", "ndcg", "judgment_coverage"]].mean())
display(lesson.table(pd.DataFrame(rankings), "rankings"))
sparse = sparse_embed(["Atlas E104 recovery"])[0]
tokenizer, _ = sparse_model("cpu")
expanded = sorted(zip(sparse["indices"], sparse["values"]), key=lambda p: -p[1])[:16]
display(
    lesson.table(
        pd.DataFrame(
            [
                {"vocabulary_token": tokenizer.convert_ids_to_tokens(i), "weight": w}
                for i, w in expanded
            ]
        ),
        "expansion",
    )
)
hand = dict(reciprocal_rank_fusion([["a", "b"], ["b", "c"]]))
assert abs(hand["b"] - (1 / 62 + 1 / 61)) < 1e-12
fig, ax = plt.subplots(figsize=(10, 4))
results.groupby("method")[["recall", "mrr", "ndcg"]].mean().plot.bar(ax=ax)
ax.set(
    ylabel="Mean score (answerable queries)",
    ylim=(0, 1.1),
    title="Development comparison; inspect individual misses",
)
ax.tick_params(axis="x", rotation=18)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
lesson.figure(fig)
plt.show()

## Interpretation and exercise
A retriever always returning something does not establish answerability. No-answer questions are excluded from recall/MRR/nDCG, and their returned-results flag is retained. Unknown labels count as zero under the declared convention; low judgment coverage warns that an apparent miss may reflect incomplete labels. Inspect d03's cross-product support before claiming one method is universally better.

**Exercise:** identify a query where the top document changes. Classify the cause as an exact-identifier match, semantic paraphrase, expansion error, or ambiguous/incomplete judgment. Change the candidate budget on development data and account for additional query/model cost.

Rank fusion avoids score calibration but cannot recover a document outside every candidate list. SPLADE storage and encoding cost depend on pruning. Local BM25 is the educational lexical baseline; it is not Qdrant's server-side BM25 implementation. [SPLADE model card](https://huggingface.co/prithivida/Splade_PP_en_v1), [Qdrant hybrid query reference](https://qdrant.tech/documentation/search/hybrid-queries/).

In [ ]:
manifest = lesson.finish(
    {
        "five_methods": results.method.nunique() == 5,
        "seven_development_queries": results["query"].nunique() == 7,
        "no_answer_metrics_excluded": bool(results.loc[~results.answerable, "recall"].isna().all()),
        "fusion_hand_check": abs(hand["b"] - (1 / 62 + 1 / 61)) < 1e-12,
        "review_valid": validate_gold()["human_review"] == "approved",
    },
    observations={
        "ingest_seconds": ingest_seconds,
        "query_split": "development",
        "frozen_configuration": {"k": 5, "candidates": 20, "RRF_constant": 60},
    },
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.